In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q xgboost

import os
import glob
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'
DATA_BASE = os.path.join(PROJECT_DIR, 'data/raw')
RANDOM_STATE = 42

In [ ]:
def compare_with_without_duplicates(name, df, label_col, drop_cols=None):
    print(name)
    drop_cols = drop_cols or []
    df = df.copy()

    print(df.shape[0], df.shape[1])

    n_dup_full = df.duplicated(keep=False).sum()
    print(n_dup_full, 100*n_dup_full/len(df))

    feature_cols = [c for c in df.columns if c not in drop_cols + [label_col]]
    n_dup_features_only = df.duplicated(subset=feature_cols, keep=False).sum()
    print(n_dup_features_only, 100*n_dup_features_only/len(df))

    if n_dup_features_only > 0:
        dup_mask = df.duplicated(subset=feature_cols, keep=False)
        conflicting = df[dup_mask].groupby(feature_cols)[label_col].nunique()
        n_conflicting_groups = (conflicting > 1).sum()
        if n_conflicting_groups > 0:
            print(n_conflicting_groups)

    df_dedup = df.drop_duplicates(subset=feature_cols + [label_col]).reset_index(drop=True)
    print(df_dedup.shape[0], df.shape[0], df.shape[0]-df_dedup.shape[0])

    def train_eval(data, tag):
        X = data[feature_cols].select_dtypes(include=[np.number])
        y = data[label_col]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
        )
        model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                        class_weight='balanced', n_jobs=-1)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        print(tag, len(data), acc, f1, prec, rec)
        return {'tag': tag, 'n': len(data), 'accuracy': acc, 'f1': f1,
                'precision': prec, 'recall': rec}

    res_with = train_eval(df, "with")
    res_without = train_eval(df_dedup, "without")

    drop_f1 = res_with['f1'] - res_without['f1']
    print(drop_f1)

    return {'dataset': name, 'n_full_dup': int(n_dup_full),
            'n_feature_dup': int(n_dup_features_only),
            'f1_with_dup': res_with['f1'], 'f1_without_dup': res_without['f1'],
            'f1_drop': drop_f1}

In [ ]:
cic_files = glob.glob(os.path.join(DATA_BASE, 'cic_ids_2017', '*.csv'))
df_cic = pd.concat([pd.read_csv(f, low_memory=False) for f in cic_files], ignore_index=True)
df_cic = df_cic.replace([np.inf, -np.inf], np.nan).dropna()
df_cic['label_bin'] = (df_cic[' Label'].astype(str).str.strip() != 'BENIGN').astype(int)

result_cic = compare_with_without_duplicates(
    "CIC-IDS2017", df_cic, label_col='label_bin', drop_cols=[' Label']
)

In [ ]:
df_unsw_train = pd.read_csv(os.path.join(DATA_BASE, 'unsw_nb15', 'UNSW_NB15_training-set.csv'), low_memory=False)
df_unsw_test = pd.read_csv(os.path.join(DATA_BASE, 'unsw_nb15', 'UNSW_NB15_testing-set.csv'), low_memory=False)
df_unsw = pd.concat([df_unsw_train, df_unsw_test], ignore_index=True)
df_unsw = df_unsw.replace([np.inf, -np.inf], np.nan).dropna()

result_unsw = compare_with_without_duplicates(
    "UNSW-NB15", df_unsw, label_col='label', drop_cols=['id', 'attack_cat']
)

In [ ]:
df_ctu_attack = pd.read_csv(os.path.join(DATA_BASE, 'ctu13', 'CTU13_Attack_Traffic.csv'), low_memory=False)
df_ctu_normal = pd.read_csv(os.path.join(DATA_BASE, 'ctu13', 'CTU13_Normal_Traffic.csv'), low_memory=False)
raw_label_variants = ['Label', 'LABEL', 'Label_original']
df_ctu_attack = df_ctu_attack.drop(columns=[c for c in raw_label_variants if c in df_ctu_attack.columns], errors='ignore')
df_ctu_normal = df_ctu_normal.drop(columns=[c for c in raw_label_variants if c in df_ctu_normal.columns], errors='ignore')
common_cols = df_ctu_attack.columns.intersection(df_ctu_normal.columns)
df_ctu_attack, df_ctu_normal = df_ctu_attack[common_cols].copy(), df_ctu_normal[common_cols].copy()
df_ctu_attack['label'] = 1
df_ctu_normal['label'] = 0
df_ctu = pd.concat([df_ctu_attack, df_ctu_normal], ignore_index=True)
df_ctu = df_ctu.replace([np.inf, -np.inf], np.nan).dropna()

result_ctu = compare_with_without_duplicates(
    "CTU-13", df_ctu, label_col='label', drop_cols=['Unnamed: 0']
)

In [ ]:
summary = pd.DataFrame([result_cic, result_unsw, result_ctu])
print(summary.to_string(index=False))

OUTPUT_DIR = os.path.join(PROJECT_DIR, 'notebooks/results')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'duplicate_check_summary.csv'), index=False)

for _, row in summary.iterrows():
    print(row['dataset'], row['f1_drop'])